In [60]:
from contextlib import asynccontextmanager
from typing import List
from fastapi import FastAPI, HTTPException, Request, BackgroundTasks
from schemas import TransactionSchema, PredictionResponseSchema
from starlette.concurrency import run_in_threadpool
import joblib
import pandas as pd
from uuid import uuid4
from datetime import datetime, timezone

def valid_transaction():
    return {
        "user_id": 42,
        "amount": 1500.50,
        "merchant_category": "grocery",
        "location_lat": 50.45,
        "location_lon": 30.52,
        "is_international": False,
    }

In [61]:
valid_transaction1 = valid_transaction()
valid_transaction1

{'user_id': 42,
 'amount': 1500.5,
 'merchant_category': 'grocery',
 'location_lat': 50.45,
 'location_lon': 30.52,
 'is_international': False}

In [62]:
valid_transaction2 = TransactionSchema(**valid_transaction())
pd.DataFrame(valid_transaction2)

,0,1
0,user_id,42
1,amount,1500.5
2,merchant_category,MerchantCategory.GROCERY
3,location_lat,50.45
4,location_lon,30.52
5,is_international,False


In [63]:
model = joblib.load("../models/model.joblib")
    
proba = model.predict_proba(pd.DataFrame([valid_transaction1]))
fraud_probability = proba[0][1]
is_fraud = fraud_probability >= 0.5
risk_level = "high" if fraud_probability >= 0.7 else "medium" if fraud_probability >= 0.3 else "low"

res = PredictionResponseSchema(
    transaction_id=uuid4(),
    timestamp=datetime.now(tz=timezone.utc),
    is_fraud=is_fraud,
    fraud_probability=fraud_probability,
    risk_level=risk_level
)
res

PredictionResponseSchema(transaction_id=UUID('9a6894f0-8146-4244-912c-ac3d887a1842'), timestamp=datetime.datetime(2026, 10, 6, 20, 11, 47, 608709, tzinfo=datetime.timezone.utc), is_fraud=False, fraud_probability=0.002890096517559789, risk_level='low')